# Aula 01 · Como o computador guarda números

Esta aula apresenta o [capítulo 1 do site](https://lacouth.github.io/metodos_telecom-site/unidade1-erros/01-erros/). A ideia central: **todo
número no computador é aproximado**, e todo método numérico também. Saber
**medir** o erro, decidir **quando parar** e reconhecer as armadilhas do
arredondamento — o acúmulo, o cancelamento e o estouro — é o que separa um
resultado confiável de um número bonito e errado.

**Ao fim da aula você consegue:**

1. explicar por que `0.1 + 0.2 != 0.3` e comparar floats do jeito certo;
2. calcular erro absoluto, erro relativo e o erro relativo **aproximado**, que não
   precisa da resposta exata;
3. deduzir a série de Taylor de $e^x$ e parar a soma com a tolerância de
   Scarborough;
4. reconhecer — e reescrever — contas que acumulam erro, cancelam algarismos ou
   estouram.

**Roteiro:** 🧩 · 1. exato não existe · 2. absoluto e relativo · 3. 🧑‍🏫 Taylor e a
parada · 4. acúmulo · 5. cancelamento · 6. estouro · 7. outra área · 🎯 prática ·
🧩 o relógio · 📋 a lista · 🚪

O caderno ocupa mais de um encontro: pare onde a aula terminar e continue daí na
seguinte.

## Como usar este caderno

- **Rode a célula ⚙️** logo abaixo antes de tudo (e de novo se o Colab reiniciar).
- **🧑‍🏫 No quadro:** a dedução é feita à mão, no quadro. Acompanhe **no seu
  caderno de papel** — é o mesmo tipo de conta que cai na parte em papel da prova.
  O resumo fica recolhido aqui, para conferir depois.
- **✍️ Passo:** o código é escrito ao vivo, em pedaços pequenos — a instrução
  está logo acima de cada célula vazia. Estudando sozinho, escreva você mesmo; o
  código completo está no capítulo do site (links 📖).
- Depois de escrever e **antes de rodar**, registre a sua previsão. Só então rode
  e abra o **▶ O que aconteceu**. A previsão errada é a parte que ensina — não a
  apague.
- **🎯 Sua vez:** escreva a função no lugar de `# sua solução aqui` e rode a
  célula `confere` logo abaixo: ✅ acertou, ❌ ainda não. Tente antes de abrir a
  💡 Dica.
- O caderno pode ocupar mais de uma aula: continue de onde parou, rodando antes a
  célula ⚙️ e as células 📦.

In [ ]:
# ⚙️ Rode esta célula antes de tudo. Ela prepara a correção automática dos
# exercícios 🎯 — não precisa ler (usa coisas que não fazem parte do curso).
import math


def _mostra(argumentos):
    textos = []
    for a in argumentos:
        textos.append(a.__name__ if callable(a) else repr(a))
    return ", ".join(textos)


def _numero(x):
    try:
        float(x)
        return not isinstance(x, (str, bool))
    except (TypeError, ValueError):
        return False


def _igual(veio, esperado, tol):
    # Número: compara com tolerância relativa, porque conta com float quase
    # nunca bate na última casa. Lista, tupla ou array: item a item.
    if _numero(esperado) and _numero(veio):
        return math.isclose(float(veio), float(esperado), rel_tol=tol, abs_tol=tol)
    if isinstance(esperado, (list, tuple)) and hasattr(veio, "__len__") and not isinstance(veio, str):
        if len(veio) != len(esperado):
            return False
        return all(_igual(v, e, tol) for v, e in zip(veio, esperado))
    return veio == esperado


def confere(funcao, casos, tol=1e-6):
    """Chama funcao com cada caso (argumentos, esperado) e diz se acertou."""
    certos = 0
    for numero, (argumentos, esperado) in enumerate(casos, start=1):
        chamada = f"{funcao.__name__}({_mostra(argumentos)})"
        try:
            veio = funcao(*argumentos)
        except Exception as erro:
            print(f"❌ {chamada} deu erro: {type(erro).__name__}: {erro}")
            continue
        if veio is not None and _igual(veio, esperado, tol):
            certos += 1
            print(f"✅ {chamada} devolveu {veio!r}")
        elif veio is None:
            print(f"❌ {chamada} devolveu None — faltou o return?")
        else:
            print(f"❌ {chamada} devolveu {veio!r}, mas devia ser {esperado!r}")
    print(f"{certos} de {len(casos)} certos")


def confere_valor(nome, valor, esperado, tol=1e-6):
    """Diz se a variável `nome` ficou com o valor esperado."""
    if valor is not None and _igual(valor, esperado, tol):
        print(f"✅ {nome} = {valor!r}")
    else:
        print(f"❌ {nome} vale {valor!r}, mas devia ser {esperado!r}")

# --- bibliotecas desta aula ---
import numpy as np
import matplotlib.pyplot as plt

## 🧩 O problema da aula

> **Estação meteorológica — sistemas embarcados.**
>
> *Uma estação meteorológica automática usa um microcontrolador (como o ESP32) que
> lê os sensores a cada 0,1 s e marca o horário de cada leitura somando 0,1 a um
> contador. O contador é um `float32`, porque "é o que o chip faz rápido". Depois
> de um dia ligada, a estação envia dados com **horários errados**: as chuvas
> aparecem minutos antes de terem começado. O técnico jura que o código está
> certo: "**é só somar 0,1, como pode errar?**"*

No fim da aula, você mede o estrago e conserta o relógio com uma linha.

## 1. Exato não existe

O computador guarda números em binário, com 53 algarismos binários (cerca de 16
decimais). $0{,}1$ não tem representação finita em binário — como $1/3$ não tem
em decimal —, e o resto é descartado.

📖 [capítulo 1 · Exato não existe](https://lacouth.github.io/metodos_telecom-site/unidade1-erros/01-erros/#exato-nao-existe)

**✍️ Passo 1.** Imprima `0.1 + 0.2` e depois `0.1 + 0.2 == 0.3`.

In [ ]:
# ✍️ passo 1

**Preveja:** o que sai em cada linha?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

`0.30000000000000004` e `False`. O erro está na 17.ª casa — e basta para o
`==` dizer que os números são diferentes.

</details>

> 🧰 **Python: formatando números no `print`**
>
> Dentro de uma f-string, o que vem depois dos dois-pontos diz **como** mostrar o
> número: `{x:.2f}` mostra 2 casas decimais, `{x:.20f}` mostra 20. O número antes
> do ponto é a **largura** (`{x:8.2f}` ocupa 8 caracteres, bom para alinhar
> tabelas), e `{x:.1e}` usa notação científica.

In [ ]:
# 🧰 exemplo — só rode e veja a saída
x = 3.14159
print(f"{x:.2f}")
print(f"[{x:8.2f}]")
print(f"{0.0045:.1e}")

**✍️ Passo 2.** Imprima `f"{0.1:.20f}"`.

In [ ]:
# ✍️ passo 2

**Preveja:** que número o computador guarda, de fato, quando você escreve `0.1`?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

`0.10000000000000000555`: um pouquinho **mais** que um décimo. Todo `0.1`
do seu programa já nasce com esse erro.

</details>

**✍️ Passo 3.** Some `0.1` dez vezes num laço (acumulador começando em `0.0`). Imprima o total, `total == 1.0` e `abs(total - 1.0) < 1e-9`.

In [ ]:
# ✍️ passo 3

**Preveja:** as três linhas: o que sai?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

`0.9999999999999999`, `False` e `True`. **Nunca compare floats com `==`**:
compare com uma tolerância. É assim que os testes das listas funcionam.

📖 [capítulo 1 · Exato não existe](https://lacouth.github.io/metodos_telecom-site/unidade1-erros/01-erros/#exato-nao-existe)

</details>

> ⚠️ **Armadilha.** Um `while total != 1.0:` com esse laço **nunca termina**: o total passa por
`0.9999999999999999`, vai para `1.0999999999999999` e segue para sempre.

### 🎯 Sua vez — Quantas vezes até chegar lá?

Escreva `vezes_ate(valor, alvo)`, que soma `valor` a um total começado em
`0.0` **enquanto** o total for menor que `alvo`, e devolve quantas somas
foram feitas.

Antes de rodar o `confere`: quantas vezes é preciso somar `0.1` para
chegar a `1.0`?

In [ ]:
def vezes_ate(valor, alvo):
    # sua solução aqui
    pass

In [ ]:
confere(vezes_ate, [
    ((0.5, 2.0), 4),
    ((0.1, 1.0), 11),
    ((0.1, 0.3), 3),
])

<details>
<summary><b>💡 Dica</b></summary>

Um contador e um acumulador, os dois começando em zero, e um `while total < alvo:`.

</details>

## 2. Erro absoluto e erro relativo

Erro absoluto: $\lvert \text{aprox} - \text{exato} \rvert$. Erro relativo:
o absoluto dividido pelo tamanho do exato, em %. É o relativo que diz se o erro
**importa**.

📖 [capítulo 1 · Erro absoluto e erro relativo](https://lacouth.github.io/metodos_telecom-site/unidade1-erros/01-erros/#erro-absoluto-e-erro-relativo)

**✍️ Passo 4.** Calcule os erros absoluto e relativo (%) de medir `9999` cm num vão de ponte de `10000` cm, e de medir `9` cm num rebite de `10` cm.

In [ ]:
# ✍️ passo 4

**Preveja:** os dois erros absolutos são iguais. E os relativos?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Os dois erram **1 cm**. Mas a ponte erra **0,01 %** e o rebite, **10 %**: o
mesmo centímetro é desprezível num caso e inaceitável no outro.

📖 [capítulo 1 · Erro absoluto e erro relativo](https://lacouth.github.io/metodos_telecom-site/unidade1-erros/01-erros/#erro-absoluto-e-erro-relativo)

</details>

## 3. No quadro: Taylor e o critério de parada

Muitos métodos melhoram a resposta **aos poucos**. Sem saber a resposta exata,
quando parar?

📖 [capítulo 1 · No quadro: Taylor e o critério de parada](https://lacouth.github.io/metodos_telecom-site/unidade1-erros/01-erros/#no-quadro-taylor-e-o-criterio-de-parada)

### 🧑‍🏫 No quadro — a série de Taylor e o erro aproximado

Caderno de papel aberto. No quadro:

1. a série de Taylor de $e^x$ em torno de 0 (todas as derivadas valem 1);
2. cada termo é **o anterior vezes $x/n$** — sem potência, sem fatorial;
3. o erro relativo **aproximado**, $\varepsilon_a$, comparando a soma nova com a
   anterior;
4. a tolerância de Scarborough para $n$ algarismos significativos;
5. $e^{0{,}5}$ à mão até parar, para 3 algarismos.

<details>
<summary><b>▶ O resumo do quadro</b></summary>

$$
e^x = 1 + x + \frac{x^2}{2!} + \frac{x^3}{3!} + \cdots
\qquad
\varepsilon_a = \left\lvert \frac{\text{atual} - \text{anterior}}{\text{atual}} \right\rvert \times 100\%
\qquad
\varepsilon_s = 0{,}5 \times 10^{\,2-n}\,\%
$$

Pare quando $\varepsilon_a < \varepsilon_s$. Para 3 algarismos,
$\varepsilon_s = 0{,}05\,\%$, e a série de $e^{0{,}5}$ para com **6 termos**.

</details>

**✍️ Passo 5.** Com `x = 0.5`, `soma = 1.0`, `termo = 1.0`, faça um laço `for n in range(1, 6):` que calcula `termo = termo * x / n`, soma e imprime `n + 1` e `soma`.

In [ ]:
# ✍️ passo 5

**Preveja:** com 6 termos, quantas casas batem com `np.exp(0.5)` = 1,6487212...?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

A última soma é `1.6486979...`: **quatro** algarismos certos (1,648). A série
converge depressa porque $x = 0{,}5$ é pequeno.

</details>

**✍️ Passo 6.** No mesmo laço, guarde a soma anterior antes de somar e imprima também `abs((soma - anterior) / soma) * 100` — o $\varepsilon_a$.

In [ ]:
# ✍️ passo 6

**Preveja:** em que termo $\varepsilon_a$ fica abaixo de 0,05 %?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

No **6.º** termo ($\varepsilon_a \approx 0{,}016\,\%$). E repare: o erro
**verdadeiro** nesse termo é $0{,}0014\,\%$, dez vezes menor. $\varepsilon_a$
é uma estimativa **pessimista** — por isso é segura como critério de parada.

📖 [capítulo 1 · No quadro: Taylor e o critério de parada](https://lacouth.github.io/metodos_telecom-site/unidade1-erros/01-erros/#no-quadro-taylor-e-o-criterio-de-parada)

</details>

### 🎯 Sua vez — O erro relativo aproximado

Escreva `eps_a(atual, anterior)`, que devolve o erro relativo aproximado em
%. Você vai usar esta função em raízes, sistemas e integrais.

In [ ]:
def eps_a(atual, anterior):
    # sua solução aqui
    pass

In [ ]:
confere(eps_a, [
    ((1.648437500, 1.645833333), 0.15797790331754172),
    ((10.0, 8.0), 20.0),
    ((-4.0, -5.0), 25.0),
])

<details>
<summary><b>💡 Dica</b></summary>

Divida pelo **atual**, não pelo anterior, e use `abs` no resultado inteiro.

</details>

## 4. Arredondamento que se acumula

Em 1991, uma bateria de mísseis Patriot não interceptou um Scud, e 28 soldados
morreram. O relógio contava décimos de segundo e multiplicava por $0{,}1$ guardado
em 24 bits (23 depois da vírgula binária). A bateria estava ligada havia 100 horas.

📖 [capítulo 1 · Arredondamento que se acumula](https://lacouth.github.io/metodos_telecom-site/unidade1-erros/01-erros/#arredondamento-que-se-acumula)

> 🧰 **Comando novo: `np.floor`**
>
> `np.floor(x)` arredonda **para baixo**: devolve o maior inteiro que não passa de
> `x`. É o "cortar as casas" de um registrador ou de um sistema que guarda
> centavos. Cuidado com os negativos: "para baixo" é na direção de $-\infty$.

In [ ]:
# 🧰 exemplo — só rode e veja a saída
print(np.floor(2.9))
print(np.floor(-2.1))
print(np.floor(0.1 * 2**23))

**✍️ Passo 7.** Calcule `decimo = np.floor(0.1 * 2**23) / 2**23` (o 0,1 cortado em 23 bits), o erro `0.1 - decimo` e o atraso depois de 100 horas: `100 * 3600 * 10` tiques vezes o erro.

In [ ]:
# ✍️ passo 7

**Preveja:** um erro de $10^{-7}$ s por tique vira quanto em 100 horas?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Cerca de **0,34 s**. Nesse tempo, um Scud a 1676 m/s percorre **575 m**: o
radar procurou o míssil longe de onde ele estava. Erros pequenos **de mesmo
sinal**, somados milhões de vezes, deixam de ser pequenos.

📖 [capítulo 1 · Arredondamento que se acumula](https://lacouth.github.io/metodos_telecom-site/unidade1-erros/01-erros/#arredondamento-que-se-acumula)

</details>

## 5. Cancelamento catastrófico

Subtrair dois números **quase iguais** apaga os algarismos iguais, e sobra só o que
estava contaminado. Foi o que estragou a derivada com $h$ pequeno demais.

📖 [capítulo 1 · Cancelamento catastrófico](https://lacouth.github.io/metodos_telecom-site/unidade1-erros/01-erros/#cancelamento-catastrofico)

**✍️ Passo 8.** Para $x^2 + 10^8 x + 1 = 0$, calcule a raiz pequena por Bhaskara, `(-b + np.sqrt(b**2 - 4*a*c)) / (2*a)`, com `a = 1.0`, `b = 1e8`, `c = 1.0`, e substitua na equação.

In [ ]:
# ✍️ passo 8

**Preveja:** a raiz é quase $-10^{-8}$. A conta acerta?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Sai `-7.45e-09`: **25 % de erro**, e substituída na equação dá 0,25 em vez de
zero. $-b + \sqrt{\Delta}$ subtrai dois números que concordam em quase todas
as casas.

</details>

**✍️ Passo 9.** Agora calcule a raiz **grande**, `(-b - np.sqrt(b**2 - 4*a*c)) / (2*a)`, e a pequena como `c / (a * x_grande)` (porque $x_1 x_2 = c/a$). Substitua de novo.

In [ ]:
# ✍️ passo 9

**Preveja:** e agora?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

`-1e-08`, e a equação dá praticamente zero. A matemática é a mesma; a conta no
computador, não. A saída do cancelamento é **reescrever a conta** para não
subtrair quase-iguais.

📖 [capítulo 1 · Cancelamento catastrófico](https://lacouth.github.io/metodos_telecom-site/unidade1-erros/01-erros/#cancelamento-catastrofico)

</details>

## 6. Números grandes demais

O `float` vai até cerca de $1{,}8 \times 10^{308}$; depois disso, vira `inf`.
Inteiros de tamanho fixo **dão a volta**. Em 1996, o Ariane 5 se autodestruiu 37 s
depois do lançamento porque a velocidade horizontal não coube num inteiro de 16
bits.

📖 [capítulo 1 · Números grandes demais](https://lacouth.github.io/metodos_telecom-site/unidade1-erros/01-erros/#numeros-grandes-demais)

**✍️ Passo 10.** Um chip de 16 bits guarda inteiros de −32768 a 32767. Escreva `inteiro_16_bits(valor)`, que devolve `(valor + 32768) % 65536 - 32768` (é o que o chip faz com o número), e imprima o resultado para `32767`, `32768` e `40000`.

In [ ]:
# ✍️ passo 10

**Preveja:** 32767 cabe. E 32768, que passa do limite por um só?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

32768 vira **−32768** e 40000 vira **−25536**, sem erro nenhum: o valor "deu a volta" para os negativos, como um hodômetro que passa de 99999 para 00000.
Uma velocidade positiva virou negativa. Toda conversão de tipo precisa da
pergunta: **e se não couber?**

📖 [capítulo 1 · Números grandes demais](https://lacouth.github.io/metodos_telecom-site/unidade1-erros/01-erros/#numeros-grandes-demais)

</details>

## 7. Mesmo método, outra área

Bancos guardam saldos em centavos. Todo rendimento tem de ser arredondado — ou
**cortado**. Parece detalhe.

📖 [capítulo 1 · Mesmo método, outra área](https://lacouth.github.io/metodos_telecom-site/unidade1-erros/01-erros/#mesmo-metodo-outra-area)

**✍️ Passo 11.** Com `capital = 1234.56` e `taxa = 0.005`, faça 120 meses de rendimento de dois jeitos: sem arredondar, e cortando os centavos todo mês (`np.floor(saldo * (1 + taxa) * 100) / 100`). Imprima a diferença.

In [ ]:
# ✍️ passo 11

**Preveja:** quanto a conta perde em dez anos? E um banco com um milhão de contas?

_Sua previsão:_ 

→ escreva aqui **antes** de rodar a célula acima.

<details>
<summary><b>▶ O que aconteceu</b></summary>

Cerca de **R$ 0,77** por conta — ninguém reclama. Em um milhão de contas,
**mais de R$ 770 mil** que mudaram de dono. Cortar erra **sempre para o mesmo
lado**, e erros de mesmo sinal se acumulam: é o relógio do Patriot, em
reais.

📖 [capítulo 1 · Mesmo método, outra área](https://lacouth.github.io/metodos_telecom-site/unidade1-erros/01-erros/#mesmo-metodo-outra-area)

</details>

## 🎯 Prática

Retoma o bloco *7. Mesmo método, outra área*.
📖 [capítulo 1 · Mesmo método, outra área](https://lacouth.github.io/metodos_telecom-site/unidade1-erros/01-erros/#mesmo-metodo-outra-area)

### 🎯 Sua vez — Arredondar ou cortar?

Escreva `perda_por_corte(valores)`, que recebe uma lista de valores em reais
(com mais de duas casas) e devolve **quanto se perde, no total**, cortando
os centavos (`np.floor(v * 100) / 100`) em vez de arredondar
(`np.round(v, 2)`).

In [ ]:
def perda_por_corte(valores):
    # sua solução aqui
    pass

In [ ]:
confere(perda_por_corte, [
    (([10.456, 3.999, 7.125],), 0.02000000000000135),
    (([1.004, 2.001],), 0.0),
])

<details>
<summary><b>💡 Dica</b></summary>

Acumulador: para cada valor, some a diferença entre o arredondado e o cortado.

</details>

## 🧩 Resolvendo o problema

> *"**É só somar 0,1, como pode errar?**"* — o técnico da estação meteorológica.

A célula 📦 reproduz o relógio do microcontrolador: soma `0.1` em `float32` a cada
tique. Rode e veja quanto ele marca depois de um dia.

> 🧰 **Comando novo: `np.float32`, a precisão simples**
>
> O `float` do Python tem cerca de 16 algarismos (precisão **dupla**, 64 bits).
> Microcontroladores como o do datalogger usam muitas vezes a precisão
> **simples**: 32 bits, cerca de 7 algarismos. `np.float32(x)` guarda `x` desse
> jeito, e toda conta feita com ele continua em `float32`. Compare o 0,1 guardado
> dos dois jeitos:

In [ ]:
# 🧰 exemplo — só rode e veja a saída
print(f"{0.1:.20f}")
print(f"{np.float32(0.1):.20f}")

In [ ]:
# 📦 dados prontos — só rode esta célula
# O relógio do datalogger, como o microcontrolador faz: soma 0,1 s em float32
# (precisão simples, cerca de 7 algarismos) a cada leitura.
def relogio_float32(tiques):
    t = np.float32(0.0)
    passo = np.float32(0.1)
    for i in range(tiques):
        t = t + passo
    return float(t)


UM_DIA = 864000          # tiques de 0,1 s em 24 horas
print("depois de 1 dia, o relógio marca", relogio_float32(UM_DIA), "s")
print("deveria marcar                   ", UM_DIA / 10, "s")

<details>
<summary><b>▶ Por que tanto erro?</b></summary>

O `float32` tem só **24 algarismos binários** (uns 7 decimais). Quando o contador
passa de alguns milhares de segundos, a distância entre dois `float32` vizinhos fica
maior que um milésimo de segundo; a soma de `0.1` é **arredondada** a cada tique, e
sempre um pouco para o mesmo lado durante longos trechos. Depois de 864 mil somas,
o relógio adianta **mais de 12 minutos**.

É o Patriot outra vez: um erro pequeno, de mesmo sinal, repetido muitas vezes.

</details>

### 🎯 Sua vez — O relógio consertado

A correção é a mesma do Patriot: **contar os tiques como inteiro** (inteiro
não tem erro de arredondamento) e converter para segundos **uma vez só**, na
hora de usar. Escreva `horario(tiques)`, que devolve o horário em segundos a
partir do número inteiro de tiques de 0,1 s.

In [ ]:
def horario(tiques):
    # sua solução aqui
    pass

In [ ]:
confere(horario, [
    ((864000,), 86400.0),
    ((36000,), 3600.0),
    ((7,), 0.7),
])

<details>
<summary><b>💡 Dica</b></summary>

Uma divisão só: `tiques / 10`. O erro fica no último algarismo, uma vez, e não se acumula.

</details>

Compare os dois relógios depois de um dia:

In [ ]:
print("float32 somando:", relogio_float32(UM_DIA), "s")
print("inteiro + 1 conta:", horario(UM_DIA), "s")

## 📋 A lista

Abra a [Lista 01](https://lacouth.github.io/metodos_telecom-site/listas/lista01/). O **Exercício 01** é à mão (✏️): as aproximações
históricas de $\pi$, $22/7$ e $355/113$. Comece por ele, no papel.

**a)** Para $22/7$, qual o erro relativo percentual?

<details>
<summary><b>▶ Resposta</b></summary>

$22/7 = 3{,}142857$; o erro absoluto é $0{,}001264$; dividido por $\pi$ e vezes
100: $\varepsilon_t \approx 0{,}0402\,\%$.

</details>

**b)** Isso garante 3 algarismos significativos? E 4?

<details>
<summary><b>▶ Resposta</b></summary>

3 sim ($0{,}0402\,\% < \varepsilon_s = 0{,}05\,\%$); 4 não
($\varepsilon_s = 0{,}005\,\%$). $22/7$ acerta $\pi$ em 3,14.

</details>

Faça a mesma conta para $355/113$ e siga para o **Exercício 02**.

## 🚪 Antes de sair

**1.** Por que o critério de parada usa $\varepsilon_a$ e não $\varepsilon_t$?

<details>
<summary><b>▶ Resposta da 1</b></summary>

Porque $\varepsilon_t$ precisa da resposta exata — e quem tem a resposta exata não
precisa do método. $\varepsilon_a$ só usa as duas últimas aproximações, e costuma
ser pessimista (maior que o erro verdadeiro), o que o torna seguro.

</details>

**2.** Você precisa calcular $\sqrt{x+1} - \sqrt{x}$ para $x = 10^{12}$. Que armadilha você espera, e como reescrever a conta?

<details>
<summary><b>▶ Resposta da 2</b></summary>

Cancelamento: as duas raízes são quase iguais. Multiplicando e dividindo pelo
conjugado, $\sqrt{x+1} - \sqrt{x} = \dfrac{1}{\sqrt{x+1} + \sqrt{x}}$ — uma
**soma**, sem cancelamento.

</details>

**3.** O relógio do datalogger usava `float32` "porque é mais rápido". Trocar para `float64` resolve?

<details>
<summary><b>▶ Resposta da 3</b></summary>

**Adia**, não resolve: o `float64` também não guarda 0,1 exatamente, e o erro
continua se acumulando — só que bem mais devagar (cerca de meio microssegundo por
dia). A correção de verdade é não acumular: contar inteiros e converter uma vez.

</details>

## 🏠 Para casa

- Refaça no papel a série de $e^{0{,}5}$ até parar com 3 algarismos.
- Termine a [Lista 01](https://lacouth.github.io/metodos_telecom-site/listas/lista01/).
- Leia o começo do [capítulo 2](https://lacouth.github.io/metodos_telecom-site/unidade2-derivadas/02-diferencas-finitas/):
  o cancelamento vai voltar, dentro da derivada numérica.